# Eje 6b — Binomial vs Black–Scholes (y mercado)

Derivados Financieros — QUANt UCEMA

Cerramos el puente discreto→continuo y miramos un contrato de mercado vs el modelo
(usando `Codigo.data.market_data`, igual que el eje 4).

| Notebook | Contenido |
|----------|-----------|
| **06a** | Fórmula BS |
| **06b** (este) | Convergencia bin↔BS + mid de mercado vs BS |
| **07** | Griegas |

## 1) Setup

In [1]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "Codigo" / "pricing").exists():
        sys.path.insert(0, str(p))
        break

from Codigo.pricing import opcion_europea_bs, opcion_europea_bin
from Codigo.data.market_data import get_spot, get_expirations, get_options_chain

print("OK")

OK


## 2) Misma opción: BS vs binomial para varios $N$

In [2]:
tipo, S, K, T, r, sigma, div = "C", 100.0, 100.0, 1.0, 0.05, 0.25, 0.0
bs = opcion_europea_bs(tipo, S, K, T, r, sigma, div)
rows = []
for n in [10, 25, 50, 100, 250, 500, 1000, 2000]:
    b = opcion_europea_bin(tipo, S, K, T, r, sigma, div, n)
    rows.append({"N": n, "binomial": b, "BS": bs, "diff": b - bs})
df = pd.DataFrame(rows)
display(df.round(6))
print(f"Con N=2000, |diff| = {abs(df.iloc[-1]['diff']):.6f}")

,N,binomial,BS,diff
0,10,12.092254,12.335999,-0.243745
1,25,12.426708,12.335999,0.090709
2,50,12.286685,12.335999,-0.049314
3,100,12.311310,12.335999,-0.024689
4,250,12.326116,12.335999,-0.009883
5,500,12.331056,12.335999,-0.004943
6,1000,12.333527,12.335999,-0.002472
7,2000,12.334763,12.335999,-0.001236


Con N=2000, |diff| = 0.001236


## 3) Un contrato de mercado vs BS

Tomamos un ticker líquido (US), el primer vencimiento, strike ATM, mid ≈ (bid+ask)/2,
e IV del panel (si viene) para repricear con BS.

Si Yahoo/limitaciones de red fallan, el notebook no revienta: muestra el error y sigue.

In [3]:
TICKER = "MSFT"
try:
    spot = get_spot(TICKER)
    exps = get_expirations(TICKER)
    expiry = exps[0]
    chain = get_options_chain(TICKER, expiry)
    print(f"{TICKER} spot={spot:.2f}  expiry={expiry}  chain={chain.shape}")

    df = chain.copy()
    for c in ("bid", "ask", "lastPrice", "strike", "impliedVolatility"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if {"bid", "ask"}.issubset(df.columns):
        df["mid"] = (df["bid"] + df["ask"]) / 2
    else:
        df["mid"] = df.get("lastPrice")

    # ATM call
    calls = df[df["type"].astype(str).str.lower().eq("call")].dropna(subset=["strike", "mid"])
    atm_idx = (calls["strike"] - spot).abs().idxmin()
    row = calls.loc[atm_idx]
    K_mkt = float(row["strike"])
    mid = float(row["mid"])
    iv = float(row["impliedVolatility"]) if pd.notna(row.get("impliedVolatility")) else float("nan")

    # TTM en años
    T_mkt = max((pd.Timestamp(expiry) - pd.Timestamp.today().normalize()).days / 365.0, 1 / 365)
    r_mkt, div_mkt = 0.04, 0.0
    sigma_use = iv if (iv == iv and 0.01 < iv < 3) else 0.25
    bs_mkt = opcion_europea_bs("C", spot, K_mkt, T_mkt, r_mkt, sigma_use, div_mkt)
    bin_mkt = opcion_europea_bin("C", spot, K_mkt, T_mkt, r_mkt, sigma_use, div_mkt, 500)

    print(f"ATM call K={K_mkt:.2f}  mid={mid:.3f}  IV={sigma_use:.3%}")
    print(f"T={T_mkt:.3f}y  BS={bs_mkt:.3f}  Bin(N=500)={bin_mkt:.3f}  mid-BS={mid-bs_mkt:.3f}")
except Exception as e:
    print("No se pudo cargar mercado (ok en offline):", type(e).__name__, e)

MSFT spot=516.17  expiry=2026-09-28  chain=(77, 19)
ATM call K=515.00  mid=4.400  IV=20.844%
T=0.005y  BS=3.856  Bin(N=500)=3.857  mid-BS=0.544


## 4) Lectura

- Si $N$ es grande, bin ≈ BS: el modelo continuo es el límite del discreto.
- Mid − BS ≠ 0: mercado usa smile/skew, tasas/divs distintos, americanas, microestructura.
- En la app: **Modelos de Pricing** compara curvas; **Market Data** trae el panel.

**Ejercicio:** cambiá el ticker a `AAPL` y compará el sesgo mid−BS OTM vs ATM.

**Siguiente:** Eje 7 — griegas ($\Delta$, $\Gamma$, $\mathcal{V}$, $\Theta$).